In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.reorg


>DOM reorganization
output-file: core.dom.reorg
title: core.dom.reorg

This notebook provides utility functions for DOM reorganization
---

In [ ]:
# | default_exp core.dom.reorg

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import json
import copy
import os
import re
from collections.abc import Callable
from typing import Optional

from collections.abc import Iterator
import pypandoc

In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | export
def reorg_node(node, section_level: list, action: Optional[Callable] = None):
    """Reorganizes the node structure to ensure that headers are treated as sections."""
    if action:
        node = action(node)

    if isinstance(node, dict):
        for key, value in node.items():
            if isinstance(value, list):
                new_list = []
                for child in value:
                    if isinstance(child, (dict, list)):
                        new_list.append(reorg_node(child, section_level=section_level, action=action))
                    else:
                        new_list.append(child)
                    if isinstance(child, dict) and 't' in child and child['t'] == "Header":
                        child['t'] = "Section"
                        child['c'] = [
                            {
                                't': 'Header',
                                'c': [
                                    child['c'][0],
                                    child['c'][1],
                                    child['c'][2],
                                ]
                            },
                            {
                                't': 'Content',
                                'c': []
                            }
                        ]
                        section_level.append(child['c'][0]['c'][0])
                it = iter(new_list)
                new_value = []
                while True:
                    try:
                        item = next(it)
                        if isinstance(item, dict) and item.get('t') == "Section":
                            item, it = reorg_section(item, it, bIgnoreLevel=False)
                        new_value.append(item)
                    except StopIteration:
                        break
                node[key] = new_value
            elif isinstance(value, dict):
                node[key] = reorg_node(value, section_level=section_level, action=action)
    elif isinstance(node, list):
        node = [
            reorg_node(child, section_level=section_level,action=action) if isinstance(child, (dict, list)) else child
            for child in node
        ]
    return node


In [ ]:
# | export
def reorg_section(section: dict, it: Iterator, bIgnoreLevel:bool=False) -> tuple[dict, Iterator]:
    """Pack following nodes into a section until a sibling or parent section is reached.
    
    Args:
        section: Section node being populated.
        it: Iterator over remaining sibling nodes.
        bIgnoreLevel: Whether to ignore section-level boundary checks.
    
    Returns:
        tuple[dict, Iterator]: Updated section and iterator for the remaining nodes.
    """

    assert section.get('t') == "Section", "Expected a Section node"
    level = section['c'][0]['c'][0]

    while True:
        try:
            item = next(it)
            if not bIgnoreLevel and isinstance(item, dict) and item.get('t') == "Section":
                # If the item is a Section, add the next items into its Content until the next Section
                next_level = item['c'][0]['c'][0]
                if next_level >= level + 1:  # lower levels need to be continuous
                    # If the next Section is at a lower level, step into further subsection handling
                    item, it = reorg_section(item, it, bIgnoreLevel=False)
                elif next_level == level:  # lower levels need to be continuous
                    # If the next Section is at the same level, we can stop here at the current recursion level
                    it = iter([item] + list(it))
                    return section, it
                    # item, it = reorg_section(item, it)
                elif next_level < level:  # higher levels don't need to be continuous
                    # If the next Section is at a higher level, we can stop here at the current recursion level
                    it = iter([item] + list(it))
                    return section, it
                else:  # next_level >= level + 2:
                    raise ValueError(f"Unexpected Section level encountered: node: {item}")

            # If the item is not a Section, we can add it to the Content of the Section
            assert section['c'][1].get('t') == "Content", "Expected a Content node"
            section['c'][1]['c'].append(item)

        except StopIteration:
            break
    return section, it


In [ ]:
# | export
def reorg_slides(raw_json: str, raw_markdown: str, slide_splitter: str = r"(^<!--\s*Slide number:\s*\d+\s*-->$)") -> str:
    """Split slide-delimited markdown into section-shaped AST nodes.
    
    Args:
        slide_splitter: Regular expression used to detect slide separators.
    
    Returns:
        str: Reorganized AST serialized as JSON.
    """

    assert raw_json or raw_markdown, "raw_json/raw_markdown content is empty. Cannot reorganize slides."
    # Split the raw_markdown into slides
    items = re.split(slide_splitter, raw_markdown, flags=re.MULTILINE)  # type: ignore
    presentation = json.loads(raw_json)  # type: ignore
    presentation['blocks'] = []  # type: ignore
    slide_header0 = {
        't': 'Section', 
        'c': [
            {
                't': 'Header', 
                'c': [
                    1,  # Header level, can be 1, 2, 3, etc.
                    ['slide header', [], []],  # The header format list [id, formtat1, format2]
                    [{'t': 'Str', 'c': 'slide header'}],  # The header content list
                ]
            },  # Header for the slide
            {
                't': 'Content', 
                'c': []
            }
            ]  # Content of the slide
        }
    # Reorganize each slide
    slide_header = None  # Initialize slide variable
    if items[0] == "":
        # If the first slide is empty, remove it
        items = items[1:]
        slide_header = copy.deepcopy(slide_header0)
        slide_header['c'][0]['c'][1][0] = '<!-- Slide number: 0 -->'
        slide_header['c'][0]['c'][2][0]['c'] = '<!-- Slide number: 0 -->'

    for item in items:
        if re.match(r"^<!--\s*Slide number:\s*\d+\s*-->$", item):
            # If the slide is a slide splitter
            slide_header = copy.deepcopy(slide_header0)  # must be deepcopy, otherwise the slide will be modified in place
            slide_header['c'][0]['c'][1][0] = item.strip()
            slide_header['c'][0]['c'][2][0]['c'] = item.strip()
        else:
            assert slide_header, f"Slide is not defined!"
            raw_ast = pypandoc.convert_text(item.strip(), "json", "md")
            ast = json.loads(raw_ast)
            assert ast.get('blocks'), f"AST blocks are not defined in {ast}"
            it = iter(ast['blocks'])
            slide, it = reorg_section(slide_header, it, bIgnoreLevel=True)
            presentation['blocks'].append(slide)

    # Convert the list of slides back to a single JSON object
    return json.dumps(presentation, ensure_ascii=False).encode("utf-8").decode("utf-8")


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()